# Harmonized 72-Class Leaf Disease Model Comparison

This notebook rebuilds the canonical 70/15/15 benchmark from the attached Kaggle dataset, then trains MobileNet V1/V2/V3 Small/V3 Large, NASNetMobile, EfficientNetB0/EfficientNetV2-B0, comparing full INT8 TFLite quality, model size, and single-thread CPU latency. CPU latency is a proxy; validate finalists on the target Android device.

Attach the multi-crop leaf disease dataset in Kaggle Notebook settings and enable a GPU accelerator.
Set `COPY_DATASET_TO_WORKING = True` in the setup cell to optionally copy the discovered raw/processed source into Kaggle working storage before benchmark preparation.


In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git'
BRANCH = 'ut'
REPO_DIR = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
COPY_DATASET_TO_WORKING = False
WORKING_DATASET_ROOT = '/kaggle/working/leaf_dataset/source'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
elif not os.path.isfile(os.path.join(REPO_DIR, 'training', 'compare_models.py')):
    raise RuntimeError('Existing repository checkout is not the updated ut pipeline; restart the Kaggle session and rerun.')
print('Repository:', REPO_DIR, 'branch:', BRANCH)
print("Copy source dataset to Kaggle working storage:", COPY_DATASET_TO_WORKING)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', os.path.join(REPO_DIR, 'requirements.txt')], check=True)
print('Dependencies installed')

In [ ]:
from pathlib import Path
import shutil
sys.path.insert(0, REPO_DIR)
from dataset.taxonomy import CANONICAL_CLASSES, canonicalize_label

def has_expected_labels(folder):
    if not folder.is_dir():
        return False
    labels = [p.name for p in folder.iterdir() if p.is_dir()]
    mapped = set()
    try:
        mapped = {canonicalize_label(label) for label in labels}
    except ValueError:
        return False
    return len(mapped) == 72

def is_valid_source_root(folder):
    folder = Path(folder)
    split_dirs = [folder / name for name in ("train", "val", "test")]
    present_splits = [path.is_dir() for path in split_dirs]
    if any(present_splits):
        return all(present_splits) and all(has_expected_labels(path) for path in split_dirs)
    return has_expected_labels(folder)

def _kaggle_mount_roots(input_root):
    input_root = Path(input_root)
    if not input_root.is_dir():
        raise FileNotFoundError(f"Kaggle input directory is missing: {input_root}")

    roots = []
    for item in sorted(input_root.iterdir()):
        if not item.is_dir():
            continue
        if item.name.lower() == "datasets":
            for owner in sorted(item.iterdir()):
                if owner.is_dir():
                    roots.extend(child for child in sorted(owner.iterdir()) if child.is_dir())
        else:
            roots.append(item)
    return roots

def find_dataset_source(input_root="/kaggle/input"):
    relative_candidates = (
        ("dataset", "raw"),
        ("raw",),
        ("data", "raw"),
        ("dataset", "processed"),
        ("processed",),
        ("data", "processed"),
        ("dataset",),
        ("data",),
        (),
    )
    seen = set()
    for mount in _kaggle_mount_roots(input_root):
        for parts in relative_candidates:
            candidate = mount.joinpath(*parts)
            key = str(candidate.resolve())
            if key in seen:
                continue
            seen.add(key)
            if is_valid_source_root(candidate):
                return candidate
    raise FileNotFoundError(
        "Could not find a raw, processed, split, or flat source root containing "
        f"the canonical 72 classes under {input_root}."
    )

SOURCE_DIR = find_dataset_source()
if COPY_DATASET_TO_WORKING:
    working_dataset_root = Path(WORKING_DATASET_ROOT)
    if working_dataset_root.exists():
        if not is_valid_source_root(working_dataset_root):
            raise ValueError(
                f"Existing dataset copy is invalid or incomplete: {working_dataset_root}"
            )
        print('Reusing valid source copy:', working_dataset_root)
    else:
        working_dataset_root.parent.mkdir(parents=True, exist_ok=True)
        print('Copying source dataset to Kaggle working storage...')
        shutil.copytree(SOURCE_DIR, working_dataset_root)
        if not is_valid_source_root(working_dataset_root):
            raise ValueError(
                f"Copied dataset failed canonical source validation: {working_dataset_root}"
            )
        print('Source copy complete:', working_dataset_root)
    SOURCE_DIR = working_dataset_root
BENCHMARK_DIR = Path('/kaggle/working/harmonized_benchmark')
print('Source:', SOURCE_DIR)
print('Canonical classes:', len(CANONICAL_CLASSES))

In [ ]:
# The prepare_benchmark step scans and rewrites every image, so it only
# runs once per session. If the benchmark already exists it is reused;
# set LEAF_USE_DEDUP_BENCHMARK=0 to refuse preparing it again.
USE_DEDUP_BENCHMARK = os.environ.get('LEAF_USE_DEDUP_BENCHMARK', '1').strip().lower() in {'1', 'true', 'yes', 'on'}
if BENCHMARK_DIR.exists():
    print('Reusing existing benchmark:', BENCHMARK_DIR)
elif USE_DEDUP_BENCHMARK:
    subprocess.run([sys.executable, os.path.join(REPO_DIR, 'dataset', 'prepare_benchmark.py'), '--data-dir', str(SOURCE_DIR), '--output-dir', str(BENCHMARK_DIR), '--seed', '42'], check=True)
    print('Prepared benchmark at', BENCHMARK_DIR)
else:
    raise RuntimeError(
        'No benchmark exists at ' + str(BENCHMARK_DIR) + '. '
        'Set LEAF_USE_DEDUP_BENCHMARK=1 or build the benchmark first.'
    )

In [ ]:
COMPARISON_DIR = Path('/kaggle/working/leaf_model_comparison')
if COMPARISON_DIR.exists():
    raise FileExistsError(f'Comparison output already exists: {COMPARISON_DIR}. Restart the session or set a new output path.')
runtime_environment = dict(os.environ)
runtime_environment['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
subprocess.run([sys.executable, os.path.join(REPO_DIR, 'training', 'compare_models.py'), '--data-dir', str(BENCHMARK_DIR), '--output-dir', str(COMPARISON_DIR), '--platform', 'kaggle'], check=True, env=runtime_environment)
print('Comparison report:', COMPARISON_DIR / 'model_comparison.json')

In [ ]:
# Pack the comparison outputs so the whole report tree can be pulled
# down as a single file.
import os
import shutil

archive_base = os.path.join('/kaggle/working', 'harmonized_comparison')
archive_path = shutil.make_archive(archive_base, 'zip', str(COMPARISON_DIR))
print("Archive:", archive_path)

In [ ]:
# Download link: the Output panel button silently refuses files over
# ~20 MB. FileLink serves the zip from this live session instead, and
# resolves both the existence check and the rendered link against the
# kernel working directory -- chdir to the archive's folder first so a
# stray CWD change cannot break the display.
# Works only while the session runs; otherwise use the Kaggle API.
import os
from IPython.display import FileLink, display

os.chdir(os.path.dirname(archive_path))

print(f"Archive size: {os.path.getsize(archive_path) / 1e6:.1f} MB")
display(FileLink(os.path.basename(archive_path)))  # click to download the zip

# Belt and braces: the panel also refuses every single file over ~20 MB, so
# split the archive into ~19 MB parts it will serve. Rejoin them afterwards
# with: copy /b <stem>.part00+<stem>.part01+... <name>.zip   (Windows)
#       cat <stem>.part* > <name>.zip                       (macOS/Linux)
MAX_PART_BYTES = 19 * 1024 * 1024
archive_name = os.path.basename(archive_path)
if os.path.getsize(archive_path) > MAX_PART_BYTES:
    stem = archive_name[: -len(".zip")]
    prefix = stem + ".part"
    for entry in os.listdir("."):
        if entry.startswith(prefix) and entry[len(prefix) :].isdecimal():
            os.remove(entry)
    with open(archive_path, "rb") as payload:
        part_index = 0
        while True:
            chunk = payload.read(MAX_PART_BYTES)
            if not chunk:
                break
            part_name = f"{prefix}{part_index:02d}"
            with open(part_name, "wb") as part_file:
                part_file.write(chunk)
            print(f"{part_name}: {len(chunk) / 1e6:.1f} MB")
            display(FileLink(part_name))
            part_index += 1
    parts = sorted(
        entry
        for entry in os.listdir(".")
        if entry.startswith(prefix) and entry[len(prefix) :].isdecimal()
    )
    print("Rejoin with: copy /b " + "+".join(parts) + f" {archive_name}  (Windows)")
    print(f"or: cat {prefix}* > {archive_name}  (macOS/Linux)")


In [ ]:
import json
import pandas as pd

report = json.loads((COMPARISON_DIR / 'model_comparison.json').read_text())
display(pd.read_csv(COMPARISON_DIR / 'model_comparison.csv').sort_values(['int8_macro_f1', 'int8_mean_inference_ms'], ascending=[False, True]))
print('Pareto frontier:', report['pareto_frontier'])
print(report['latency_note'])

**Outputs**

**Downloading the archive:** Kaggle's Output panel only handles files up to
~20 MB, so a large archive cannot be pulled from that button. The cell above
renders a `FileLink` that streams the zip directly from the live session - click
it while the session is still running (`Save & Run All` versions and stopped
sessions cannot serve it). For committed versions, pull the output with the
Kaggle API instead:

```bash
kaggle kernels output <your-username>/<notebook-slug> -p ./kaggle_out
```

If the archive is larger than ~20 MB, the cell also writes `<stem>.part00`-style splits that the panel will serve; rejoin them with `copy /b <stem>.part00+... <name>.zip` (Windows) or `cat <stem>.part* > <name>.zip` (macOS/Linux).
